# 04 — Cell-Type Enrichment Analysis

Are mechanosensitive ion channels preferentially expressed in particular cell types?

We use spatial correlation between channel genes and canonical cell-type markers
across brain regions as a proxy for cell-type enrichment. High correlation between
a channel gene and a cell-type marker set across regions suggests co-localization.

**Cell types examined:** PV interneurons, SST interneurons, VIP interneurons,
pyramidal/excitatory neurons, astrocytes, oligodendrocytes, microglia.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import gene_lists, data_loader, cell_type_markers

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

## 1. Load expression data

In [ ]:
# Mechanosensitive channel genes
channel_genes = list(gene_lists.ALL_MECHANOSENSITIVE.keys())

# Cell-type marker genes
marker_genes = cell_type_markers.ALL_MARKER_GENES

print(f"Channel genes: {len(channel_genes)}")
print(f"Marker genes: {len(marker_genes)}")
print(f"\nCell types and markers:")
for ct, markers in cell_type_markers.CELL_TYPE_MARKERS.items():
    print(f"  {ct}: {', '.join(markers)}")

In [ ]:
# Load averaged expression across donors
all_genes = list(set(channel_genes + marker_genes))
region_expr = data_loader.get_microarray_region_means(all_genes)

# Split into channel and marker expression
channel_expr = region_expr.loc[region_expr.index.isin(channel_genes)]
marker_expr = region_expr.loc[region_expr.index.isin(marker_genes)]

print(f"Channel expression: {channel_expr.shape}")
print(f"Marker expression: {marker_expr.shape}")
print(f"Shared regions: {len(channel_expr.columns.intersection(marker_expr.columns))}")

## 2. Cell-type enrichment scores

Mean spatial Spearman correlation between each channel gene and each cell type's marker gene set.

In [ ]:
enrichment = cell_type_markers.compute_celltype_enrichment_scores(
    channel_expr, marker_expr, method='spearman'
)
print(f"Enrichment matrix: {enrichment.shape}")
enrichment.head(10)

In [ ]:
fig, ax = cell_type_markers.plot_celltype_enrichment(
    enrichment,
    title='Cell-Type Enrichment of Mechanosensitive Channels\n(Spatial Spearman Correlation)',
    figsize=(14, 12),
)
plt.savefig('../figures/celltype_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. FUS-core genes: cell-type enrichment

In [ ]:
core_genes = list(gene_lists.FUS_CORE_GENES.keys())
core_enrichment = enrichment.loc[enrichment.index.isin(core_genes)]

fig, ax = cell_type_markers.plot_celltype_enrichment(
    core_enrichment,
    title='FUS Core Channels — Cell-Type Enrichment',
    figsize=(12, 7),
)
plt.savefig('../figures/celltype_enrichment_core.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Detailed channel × marker correlation matrix

In [ ]:
corr_df, pval_df = cell_type_markers.compute_spatial_correlation(
    channel_expr, marker_expr, method='spearman'
)
print(f"Correlation matrix: {corr_df.shape}")

# Count significant correlations
n_sig = (pval_df < 0.05).sum().sum()
n_total = pval_df.size
print(f"Significant correlations (p<0.05): {n_sig}/{n_total} ({100*n_sig/n_total:.1f}%)")

In [ ]:
fig, ax = cell_type_markers.plot_correlation_detail(
    corr_df, pval_df,
    title='Mechanosensitive Channel × Cell-Type Marker Spatial Correlation',
    figsize=(18, 14),
)
plt.savefig('../figures/channel_marker_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Strongest cell-type associations

Which channel genes have the strongest positive or negative correlation with each cell type?

In [ ]:
for cell_type in enrichment.columns:
    col = enrichment[cell_type].sort_values(ascending=False)
    print(f"\n{'='*60}")
    print(f"{cell_type}")
    print(f"{'='*60}")
    print(f"\nTop 5 positively correlated channels:")
    for gene, val in col.head(5).items():
        print(f"  {gene_lists.get_display_name(gene):>35s}: ρ = {val:+.3f}")
    print(f"\nTop 5 negatively correlated channels:")
    for gene, val in col.tail(5).items():
        print(f"  {gene_lists.get_display_name(gene):>35s}: ρ = {val:+.3f}")

## 6. Clustered enrichment heatmap

Cluster both channel genes and cell types to reveal patterns.

In [ ]:
# Rename index for display
plot_enrichment = enrichment.copy()
plot_enrichment.index = [gene_lists.get_display_name(g) for g in plot_enrichment.index]

# Build row colors by family
from src.clustering import build_family_palette
palette = build_family_palette(list(enrichment.index))
families = [gene_lists.get_family(g) for g in enrichment.index]
row_colors = pd.Series(
    [palette.get(f, '#95A5A6') for f in families],
    index=plot_enrichment.index,
    name='Family'
)

g = sns.clustermap(
    plot_enrichment,
    cmap='RdBu_r', center=0,
    row_colors=row_colors,
    figsize=(12, 14),
    linewidths=0.5,
    annot=True, fmt='.2f',
    method='average',
    cbar_kws={'label': 'Mean Spearman ρ'},
    dendrogram_ratio=(0.12, 0.1),
)
g.fig.suptitle('Clustered Cell-Type Enrichment of Mechanosensitive Channels', fontsize=14, y=1.02)
plt.savefig('../figures/celltype_enrichment_clustered.png', dpi=150, bbox_inches='tight')
plt.show()